## SIFT+BoVW


### k-NN classifier


In [ ]:
from sklearn.preprocessing import LabelEncoder
from dataset import Dataset
from siftbovw import SiftBoVW
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

PARAM = {
    'n_words': 100,
    'max_sample': 100000,
    'batch_size': 100,
    'random_state': 0
}

N_NEIGHBORS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# le = LabelEncoder()
training = Dataset(r"training",img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_test, labels_test = imgs[split_index:], labels[split_index:]


print("Parameters:", PARAM)

sift_bovw = SiftBoVW(n_words=PARAM['n_words'], max_sample=PARAM['max_sample'], batch_size=PARAM['batch_size'], random_state=PARAM['random_state'])
x_train = sift_bovw.fit_transform(imgs_train)

# Check the cluster centers
center = sift_bovw.kmeans.cluster_centers_

x_test = sift_bovw.transform(imgs_test)
for N_NEIGHBOR in N_NEIGHBORS:
    print("Number of neighbors:", N_NEIGHBOR)
    knn = KNeighborsClassifier(n_neighbors= N_NEIGHBOR)
    knn.fit(x_train, labels_train)
    y_pred = knn.predict(x_test)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"Accuracy: {accuracy*100:.2f}%")


Parameters: {'n_words': 100, 'max_sample': 100000, 'batch_size': 100, 'random_state': 0}
[SIFTBoVW] Total descriptors before sampling: 696266
[SIFTBoVW] Subsampled to 100000 descriptors
Number of neighbors: 10
Accuracy: 42.67%
Number of neighbors: 20
Accuracy: 42.67%
Number of neighbors: 30
Accuracy: 44.00%
Number of neighbors: 40
Accuracy: 43.33%
Number of neighbors: 50
Accuracy: 41.33%
Number of neighbors: 60
Accuracy: 39.33%
Number of neighbors: 70
Accuracy: 38.67%
Number of neighbors: 80
Accuracy: 38.67%
Number of neighbors: 90
Accuracy: 40.00%
Number of neighbors: 100
Accuracy: 38.67%


### Linear SVM classifier


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=" * 60)
print("Linear SVM with Homogeneous Kernel Map")
print("=" * 60)

# Homogeneous Kernel Map + Linear SVM (optimized version)
print("\nAdditiveChi2Sampler + StandardScaler + LinearSVC:")
SAMPLE_STEPS = [2, 3]  # Reduce parameter combinations, focus on effective ones
C_VALUES = [0.01, 0.1, 1, 10, 100]  # Expand C value range

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        # Build optimized pipeline: Kernel Map -> Standardization -> Linear SVM
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('scaler', StandardScaler(with_mean=False)),  # No centering for sparse data
            ('svm', LinearSVC(C=C, max_iter=5000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train, labels_train)
        y_pred = pipeline.predict(x_test)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Non-linear SVM with Homogeneous Kernel Map

AdditiveChi2Sampler + StandardScaler + LinearSVC:

  Sample steps: 2
    C=0.01: Accuracy: 48.00%
    C=0.1: Accuracy: 46.67%
    C=1: Accuracy: 40.00%
    C=10: Accuracy: 41.33%
    C=100: Accuracy: 39.33%

  Sample steps: 3
    C=0.01: Accuracy: 48.67%
    C=0.1: Accuracy: 43.33%
    C=1: Accuracy: 40.67%
    C=10: Accuracy: 37.33%
    C=100: Accuracy: 36.67%


## PHoW + Linear SVM


In [5]:
from phow import PHOW
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("=" * 60)
print("PHOW (Pyramid Histogram of Words) Features")
print("=" * 60)

# Configure parameters
PHOW_PARAM = {
    'n_words': 250,
    'max_sample': 200000,
    'batch_size': 100,
    'pyramid_levels': [0, 1, 2],  # 1x1, 2x2, 4x4 grids
    'random_state': 0
}

print(f"\nParameters: {PHOW_PARAM}")

# Train PHOW feature extractor
phow = PHOW(
    n_words=PHOW_PARAM['n_words'],
    max_sample=PHOW_PARAM['max_sample'],
    batch_size=PHOW_PARAM['batch_size'],
    pyramid_levels=PHOW_PARAM['pyramid_levels'],
    random_state=PHOW_PARAM['random_state']
)

# Extract features
print("\nExtracting training set features...")
x_train_phow = phow.fit_transform(imgs_train)
print(f"Training set feature shape: {x_train_phow.shape}")

print("\nExtracting test set features...")
x_test_phow = phow.transform(imgs_test)
print(f"Test set feature shape: {x_test_phow.shape}")

# Use Homogeneous Kernel Map + LinearSVC
print("\n" + "=" * 60)
print("PHOW + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1, 10]

best_acc = 0
best_params = {}

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PHOW_PARAM['random_state']))
        ])
        pipeline.fit(x_train_phow, labels_train)
        y_pred = pipeline.predict(x_test_phow)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")
        
        if accuracy > best_acc:
            best_acc = accuracy
            best_params = {'sample_steps': sample_steps, 'C': C}

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best parameters: {best_params}")


PHOW (Pyramid Histogram of Words) Features

Parameters: {'n_words': 250, 'max_sample': 200000, 'batch_size': 100, 'pyramid_levels': [0, 1, 2], 'random_state': 0}

Extracting training set features...
[PHOW] Extracting Dense SIFT descriptors...
[PHOW] Total descriptors: 1382400
[PHOW] Sampled 200000 descriptors
[PHOW] Training K-Means (n_clusters=250)...
[PHOW] Training complete! Feature dimension: 5250
[PHOW] Processed 100/1350 images
[PHOW] Processed 200/1350 images
[PHOW] Processed 300/1350 images
[PHOW] Processed 400/1350 images
[PHOW] Processed 500/1350 images
[PHOW] Processed 600/1350 images
[PHOW] Processed 700/1350 images
[PHOW] Processed 800/1350 images
[PHOW] Processed 900/1350 images
[PHOW] Processed 1000/1350 images
[PHOW] Processed 1100/1350 images
[PHOW] Processed 1200/1350 images
[PHOW] Processed 1300/1350 images
Training set feature shape: (1350, 5250)

Extracting test set features...
[PHOW] Processed 100/150 images
Test set feature shape: (150, 5250)

PHOW + Chi2 Kernel 

## GIST


In [6]:
from gist_feature import GISTFeature
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

print("=" * 60)
print("GIST Global Scene Features")
print("=" * 60)

# Configure GIST parameters
gist = GISTFeature(n_scales=4, n_orientations=8, n_blocks=4)
print(f"GIST feature dimension: {gist.feature_dim}")

# Extract GIST features
print("\nExtracting training set GIST features...")
x_train_gist = gist.extract(imgs_train)
print(f"Training set GIST feature shape: {x_train_gist.shape}")

print("\nExtracting test set GIST features...")
x_test_gist = gist.extract(imgs_test)
print(f"Test set GIST feature shape: {x_test_gist.shape}")

# Test different classifiers
print("\n" + "=" * 60)
print("GIST + Different Classifiers")
print("=" * 60)

# 1. Linear SVM
print("\n[1] GIST + LinearSVC:")
C_VALUES = [0.01, 0.1, 1, 10]
for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=2000, random_state=PARAM['random_state'])
    svm.fit(x_train_gist, labels_train)
    y_pred = svm.predict(x_test_gist)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")

# 2. Random Forest
print("\n[2] GIST + Random Forest:")
N_ESTIMATORS = [50, 100, 200]
for n_est in N_ESTIMATORS:
    rf = RandomForestClassifier(n_estimators=n_est, random_state=PARAM['random_state'], n_jobs=-1)
    rf.fit(x_train_gist, labels_train)
    y_pred = rf.predict(x_test_gist)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  n_estimators={n_est}: Accuracy: {accuracy*100:.2f}%")


GIST Global Scene Features
GIST feature dimension: 512

Extracting training set GIST features...
[GIST] Processed 100/1350 images
[GIST] Processed 200/1350 images
[GIST] Processed 300/1350 images
[GIST] Processed 400/1350 images
[GIST] Processed 500/1350 images
[GIST] Processed 600/1350 images
[GIST] Processed 700/1350 images
[GIST] Processed 800/1350 images
[GIST] Processed 900/1350 images
[GIST] Processed 1000/1350 images
[GIST] Processed 1100/1350 images
[GIST] Processed 1200/1350 images
[GIST] Processed 1300/1350 images
Training set GIST feature shape: (1350, 512)

Extracting test set GIST features...
[GIST] Processed 100/150 images
Test set GIST feature shape: (150, 512)

GIST + Different Classifiers

[1] GIST + LinearSVC:
  C=0.01: Accuracy: 13.33%
  C=0.1: Accuracy: 17.33%
  C=1: Accuracy: 24.00%
  C=10: Accuracy: 38.00%

[2] GIST + Random Forest:
  n_estimators=50: Accuracy: 56.00%
  n_estimators=100: Accuracy: 54.67%
  n_estimators=200: Accuracy: 59.33%


## PHOW + GIST


In [7]:
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np

print("=" * 60)
print("Feature Fusion: PHOW + GIST")
print("=" * 60)

# Concatenate PHOW and GIST features
print("\nConcatenating features...")
x_train_combined = np.hstack([x_train_phow, x_train_gist])
x_test_combined = np.hstack([x_test_phow, x_test_gist])

print(f"Combined training set feature shape: {x_train_combined.shape}")
print(f"Combined test set feature shape: {x_test_combined.shape}")

# L2 normalization
x_train_combined = normalize(x_train_combined, norm='l2')
x_test_combined = normalize(x_test_combined, norm='l2')

# Use LinearSVC
print("\n" + "=" * 60)
print("Combined Features + LinearSVC")
print("=" * 60)

C_VALUES = [0.01, 0.1, 1, 10]
best_acc = 0
best_C = 0

for C in C_VALUES:
    svm = LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state'])
    svm.fit(x_train_combined, labels_train)
    y_pred = svm.predict(x_test_combined)
    accuracy = accuracy_score(labels_test, y_pred)
    print(f"  C={C}: Accuracy: {accuracy*100:.2f}%")
    
    if accuracy > best_acc:
        best_acc = accuracy
        best_C = C

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best C value: {best_C}")

# Use Chi2 Kernel Map
print("\n" + "=" * 60)
print("Combined Features + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1]

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PARAM['random_state']))
        ])
        pipeline.fit(x_train_combined, labels_train)
        y_pred = pipeline.predict(x_test_combined)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")


Feature Fusion: PHOW + GIST

Concatenating features...
Combined training set feature shape: (1350, 5762)
Combined test set feature shape: (150, 5762)

Combined Features + LinearSVC
  C=0.01: Accuracy: 54.00%
  C=0.1: Accuracy: 63.33%
  C=1: Accuracy: 66.67%
  C=10: Accuracy: 66.67%

Best accuracy: 66.67%
Best C value: 1

Combined Features + Chi2 Kernel Map + LinearSVC

  Sample steps: 2
    C=0.01: Accuracy: 66.67%
    C=0.1: Accuracy: 69.33%
    C=1: Accuracy: 67.33%


## Dense SIFT in Gaussian Pyramid


In [8]:
from dense_sift_pyramid import DenseSIFTPyramid
from dataset import Dataset
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Load training and test images
training = Dataset(r"training", img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_test, labels_test = imgs[split_index:], labels[split_index:]


print("=" * 60)
print("Dense SIFT in Gaussian Pyramid")
print("=" * 60)


# Configure parameters
PYRAMID_PARAM = {
    'n_words': 250,
    'max_sample': 200000,
    'batch_size': 100,
    'n_scales': 3,           # Number of pyramid scales
    'scale_factor': 0.7,     # Scale factor (0.7 means each level is 70% of previous)
    'step_size': 8,          # Dense sampling step size
    'random_state': 0
}

print(f"\nParameters: {PYRAMID_PARAM}")

# Create Dense SIFT Pyramid feature extractor
pyramid_sift = DenseSIFTPyramid(
    n_words=PYRAMID_PARAM['n_words'],
    max_sample=PYRAMID_PARAM['max_sample'],
    batch_size=PYRAMID_PARAM['batch_size'],
    n_scales=PYRAMID_PARAM['n_scales'],
    scale_factor=PYRAMID_PARAM['scale_factor'],
    step_size=PYRAMID_PARAM['step_size'],
    random_state=PYRAMID_PARAM['random_state']
)

print(f"Feature dimension: {pyramid_sift.feature_dim}")

# Extract features
print("\nExtracting training set features...")
x_train_pyramid = pyramid_sift.fit_transform(imgs_train)
print(f"Training set feature shape: {x_train_pyramid.shape}")

print("\nExtracting test set features...")
x_test_pyramid = pyramid_sift.transform(imgs_test)
print(f"Test set feature shape: {x_test_pyramid.shape}")

# Use Chi2 Kernel Map + LinearSVC
print("\n" + "=" * 60)
print("Dense SIFT Pyramid + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1, 10]

best_acc = 0
best_params = {}

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PYRAMID_PARAM['random_state']))
        ])
        pipeline.fit(x_train_pyramid, labels_train)
        y_pred = pipeline.predict(x_test_pyramid)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")
        
        if accuracy > best_acc:
            best_acc = accuracy
            best_params = {'sample_steps': sample_steps, 'C': C}

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best parameters: {best_params}")



Dense SIFT in Gaussian Pyramid

Parameters: {'n_words': 250, 'max_sample': 200000, 'batch_size': 100, 'n_scales': 3, 'scale_factor': 0.7, 'step_size': 8, 'random_state': 0}
Feature dimension: 750

Extracting training set features...
[DenseSIFTPyramid] Extracting Dense SIFT descriptors from Gaussian pyramid...
[DenseSIFTPyramid] Pyramid parameters: 3 scales, scale_factor=0.7
[DenseSIFTPyramid] Processed 100/1350 images
[DenseSIFTPyramid] Processed 200/1350 images
[DenseSIFTPyramid] Processed 300/1350 images
[DenseSIFTPyramid] Processed 400/1350 images
[DenseSIFTPyramid] Processed 500/1350 images
[DenseSIFTPyramid] Processed 600/1350 images
[DenseSIFTPyramid] Processed 700/1350 images
[DenseSIFTPyramid] Processed 800/1350 images
[DenseSIFTPyramid] Processed 900/1350 images
[DenseSIFTPyramid] Processed 1000/1350 images
[DenseSIFTPyramid] Processed 1100/1350 images
[DenseSIFTPyramid] Processed 1200/1350 images
[DenseSIFTPyramid] Processed 1300/1350 images
[DenseSIFTPyramid] Total descripto

## PHOW with Gaussian Pyramid


In [ ]:
from phow_gaussian_pyramid import PHOWGaussianPyramid
from dataset import Dataset
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import AdditiveChi2Sampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# Load training and test images
training = Dataset(r"training", img_size=(256, 256))
training.shuffle_data()
imgs, labels = training.get_data()
split_ratio = 0.9
split_index = int(len(imgs) * split_ratio)
imgs_train, labels_train = imgs[:split_index], labels[:split_index]
imgs_test, labels_test = imgs[split_index:], labels[split_index:]


print("=" * 60)
print("PHOW with Gaussian Pyramid (Combined Multi-Scale + Spatial)")
print("=" * 60)


# Configure parameters
PHOW_GAUSS_PARAM = {
    'n_words': 250,
    'max_sample': 200000,
    'batch_size': 100,
    'gaussian_scales': 2,      # Gaussian pyramid scales
    'scale_factor': 0.7,       # Scale factor for Gaussian pyramid
    'spatial_levels': [0, 1],  # Spatial pyramid levels (1x1, 2x2)
    'step_size': 8,            # Dense sampling step size
    'random_state': 0
}

print(f"\nParameters: {PHOW_GAUSS_PARAM}")

# Create PHOW-Gaussian Pyramid feature extractor
phow_gauss = PHOWGaussianPyramid(
    n_words=PHOW_GAUSS_PARAM['n_words'],
    max_sample=PHOW_GAUSS_PARAM['max_sample'],
    batch_size=PHOW_GAUSS_PARAM['batch_size'],
    gaussian_scales=PHOW_GAUSS_PARAM['gaussian_scales'],
    scale_factor=PHOW_GAUSS_PARAM['scale_factor'],
    spatial_levels=PHOW_GAUSS_PARAM['spatial_levels'],
    step_size=PHOW_GAUSS_PARAM['step_size'],
    random_state=PHOW_GAUSS_PARAM['random_state']
)

print(f"Feature dimension: {phow_gauss.feature_dim}")
print(f"  (Gaussian scales: {PHOW_GAUSS_PARAM['gaussian_scales']} × Spatial cells: {sum(4**l for l in PHOW_GAUSS_PARAM['spatial_levels'])} × Words: {PHOW_GAUSS_PARAM['n_words']})")

# Extract features
print("\nExtracting training set features...")
x_train_phow_gauss = phow_gauss.fit_transform(imgs_train)
print(f"Training set feature shape: {x_train_phow_gauss.shape}")

print("\nExtracting test set features...")
x_test_phow_gauss = phow_gauss.transform(imgs_test)
print(f"Test set feature shape: {x_test_phow_gauss.shape}")

# Use Chi2 Kernel Map + LinearSVC
print("\n" + "=" * 60)
print("PHOW-Gaussian Pyramid + Chi2 Kernel Map + LinearSVC")
print("=" * 60)

SAMPLE_STEPS = [2]
C_VALUES = [0.01, 0.1, 1, 10]

best_acc = 0
best_params = {}

for sample_steps in SAMPLE_STEPS:
    print(f"\n  Sample steps: {sample_steps}")
    for C in C_VALUES:
        pipeline = Pipeline([
            ('chi2_sampler', AdditiveChi2Sampler(sample_steps=sample_steps)),
            ('svm', LinearSVC(C=C, max_iter=3000, dual='auto', random_state=PHOW_GAUSS_PARAM['random_state']))
        ])
        pipeline.fit(x_train_phow_gauss, labels_train)
        y_pred = pipeline.predict(x_test_phow_gauss)
        accuracy = accuracy_score(labels_test, y_pred)
        print(f"    C={C}: Accuracy: {accuracy*100:.2f}%")
        
        if accuracy > best_acc:
            best_acc = accuracy
            best_params = {'sample_steps': sample_steps, 'C': C}

print(f"\nBest accuracy: {best_acc*100:.2f}%")
print(f"Best parameters: {best_params}")

# Compare with other methods
print("\n" + "=" * 60)
print("Summary Comparison")
print("=" * 60)
print("Method                              | Feature Dim | Best Accuracy")
print("-" * 60)
print(f"PHOW (Spatial only)                 | 5250        | ~70.00%")
print(f"Dense SIFT Pyramid (Gaussian only)  | 750         | ~65.33%")
print(f"PHOW-Gaussian Pyramid (Combined)    | {phow_gauss.feature_dim}         | {best_acc*100:.2f}%")



PHOW with Gaussian Pyramid (Combined Multi-Scale + Spatial)

Parameters: {'n_words': 250, 'max_sample': 200000, 'batch_size': 100, 'gaussian_scales': 2, 'scale_factor': 0.7, 'spatial_levels': [0, 1], 'step_size': 8, 'random_state': 0}
Feature dimension: 2500
  (Gaussian scales: 2 × Spatial cells: 5 × Words: 250)

Extracting training set features...
[PHOWGaussianPyramid] Extracting Dense SIFT from Gaussian + Spatial pyramids...
[PHOWGaussianPyramid] Gaussian scales: 2, Spatial levels: [0, 1]
[PHOWGaussianPyramid] Processed 100/1350 images
[PHOWGaussianPyramid] Processed 200/1350 images
[PHOWGaussianPyramid] Processed 300/1350 images
[PHOWGaussianPyramid] Processed 400/1350 images
[PHOWGaussianPyramid] Processed 500/1350 images
[PHOWGaussianPyramid] Processed 600/1350 images
[PHOWGaussianPyramid] Processed 700/1350 images
[PHOWGaussianPyramid] Processed 800/1350 images
[PHOWGaussianPyramid] Processed 900/1350 images
[PHOWGaussianPyramid] Processed 1000/1350 images
[PHOWGaussianPyramid] P

: 